In [1]:
import pandas as pd

In [2]:
# read all excel files in a path

import os
import glob

path = r'data/final/tables/annotations/filled' # use your path
all_files = glob.glob(os.path.join(path, "*.xlsx"))

df_from_each_file = (pd.read_excel(f, skiprows = 3) for f in all_files)
df = pd.concat(df_from_each_file, ignore_index=True)

#df = pd.read_excel(os.path.join(path, "annotation_sebastian.xlsx"), skiprows = 3)

c:\Users\LENOVO\anaconda3\envs\bp_digitalization\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\Users\LENOVO\anaconda3\envs\bp_digitalization\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\Users\LENOVO\anaconda3\envs\bp_digitalization\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


In [3]:
df

,id,filename,met,value,Wert korrekt? (Ja/ Nein),Korrigierte Wert (falls nötig),Alternativbegriff/ Umschreibung,Standardisierte Fehlerquelle,Begründung,Notiz,Alternativbegriff/Umschreibung,Wert korrekt? (Nein)
0,17,17_0.jpg,grz_value,0.2,Ja,NaN,NaN,NaN,NaN,(In Festsetzung),NaN,NaN
1,17,17_0.jpg,gfz_value,NaN,Nein,NaN,Geschoßflächenzahl = Geschossflächenzahl,4: Begriff nicht gefunden,"Geschoßflächenzahl mit ""ß"" statt ""ss""",(In Festsetzung),NaN,NaN
2,X_17,17_0.jpg,gfz_value,NaN,NaN,0.2,NaN,NaN,NaN,(In Festsetzung),NaN,NaN
3,17,17_0.jpg,hw100_value,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,17,17_0.jpg,hw10_value,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
3649,4781,4781_9.jpg,eg_fok_unit,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3650,4781,4781_9.jpg,gok_value,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3651,4781,4781_9.jpg,gok_unit,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3652,4781,4781_9.jpg,fok_value,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
df.dropna(subset=['id'], inplace=True)

In [5]:
# Keeping only evaluation of existing rows 

df['id'] = df['id'].astype(str)
df = df[~df['id'].str.startswith('X_')]

In [6]:
def check_correct_result(df):
    """
    Groups by 'id' and checks for the following conditions:
    - If at least one 'Ja' and at least one 'Nein' exists -> 'failed to extract all correct metrics'
    - If at least one 'Ja' exists -> 'extracted all correct metrics'
    - If all values are NaN -> 'no extracted metric'
    - Otherwise -> 'incorrect metric'

    Args:
    df (pd.DataFrame): The input DataFrame.

    Returns:
    pd.DataFrame: A DataFrame with 'id' and 'correct_result'.
    """
    def determine_result(group):
        has_ja = group['Wert korrekt? (Ja/ Nein)'].eq('Ja').any()
        has_nein = group['Wert korrekt? (Ja/ Nein)'].eq('Nein').any()
        all_na = group['Wert korrekt? (Ja/ Nein)'].isna().all()

        if has_ja and has_nein:
            return 'Failed extraction: LLM failed to extract all of the metrics correctly'
        elif has_ja:
            return 'Correct extraction: LLM extracted all metrics correctly'
        elif has_nein:
            return 'Failed extraction: LLM failed to extract any of the metrics correctly'
        elif all_na:
            return 'Correct extraction: no extracted metric'

    # Group by 'id' and apply the function
    result_df = df.groupby('id').apply(determine_result).reset_index(name='correct_result')
    
    return result_df


In [7]:
def evaluate_llm_performance_on_data(df):

    metrics_evaluation = []

    for metric in df['met'].unique():

        keyword_subset = df[df['met'] == metric].copy()  #

        keyword_subset['value_match'] = keyword_subset['Wert korrekt? (Ja/ Nein)'].apply(lambda x: 1 if x == 'Ja' or pd.isna(x) else 0)

        evaluation_results = check_correct_result(keyword_subset).value_counts('correct_result').reset_index()

        evaluation_results['metric'] = metric

        metrics_evaluation.append(evaluation_results)
    
    return pd.concat(metrics_evaluation, axis=0)
    

In [8]:
evaluate_data = evaluate_llm_performance_on_data(df)

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_32244\2081264604.py:30: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  result_df = df.groupby('id').apply(determine_result).reset_index(name='correct_result')
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_32244\2081264604.py:30: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  result_df = df.groupby('id').apply(determine_result).reset_index(name='correct_result')
C:\U

In [9]:
evaluate_data

,correct_result,count,metric
0,Correct extraction: no extracted metric,30,grz_value
1,Correct extraction: LLM extracted all metrics ...,6,grz_value
2,Failed extraction: LLM failed to extract any o...,5,grz_value
3,Failed extraction: LLM failed to extract all o...,1,grz_value
0,Correct extraction: no extracted metric,31,gfz_value
1,Failed extraction: LLM failed to extract any o...,7,gfz_value
2,Correct extraction: LLM extracted all metrics ...,4,gfz_value
0,Correct extraction: no extracted metric,40,hw100_value
1,Correct extraction: LLM extracted all metrics ...,2,hw100_value
0,Correct extraction: no extracted metric,42,hw10_value


In [10]:
evaluate_data_pivot = evaluate_data.pivot(index='correct_result', columns='metric', values='count').fillna(0)

In [11]:
evaluate_data_pivot.loc["Total"] = evaluate_data_pivot.sum()


In [12]:
evaluate_data_pivot[['gfz_value']]

metric,gfz_value
correct_result,
Correct extraction: LLM extracted all metrics correctly,4.0
Correct extraction: no extracted metric,31.0
Failed extraction: LLM failed to extract all of the metrics correctly,0.0
Failed extraction: LLM failed to extract any of the metrics correctly,7.0
Total,42.0
